<h1>General notes on how ptypy works</h1>

Ptypy has capabilities for both performing actual reconstructions from data collected experimentally as well as for simulating a whole ptychograpic experiment. All examples of reconstruction scripts that are found under the /ptypy/templates uses simulated data only.
While these templates often use a .ptyd file to load the data, you would probably have your experimental data gathered in h5 files. To see examples using experimental data, look at the tutorials located at https://ptycho.github.io/tutorials/intro.html instead.



<h2>Creating your reconstruction script</h2>

Your reconstruction script will consist of X parts, importing all the neccessary modules, hardcoded info such as the paths to you files, the reconstruction parameters that will all be stored in the so called parameter tree. All possible parameters have a default value which will be used if you don't specify another value for it. The parameter tree works similarly to a nested dictionary. Let's start with importing some modules and define some hardcoded info that we'll use later on!



In [7]:
import ptypy
from ptypy import utils as u
from ptypy import defaults_tree
ptypy.load_ptyscan_module("nanomax")

# Hardcoded info
scannr = 0    # Scan number created by the beamline. There is no need to specify the possible preceeding zeros of the scan number.

# Where to store the different output files
out_dir = f'../my_reconstruction_{scannr:06d}'
out_dir_data = out_dir + 'data/'
out_dir_dumps = out_dir + 'dumps/'
out_dir_rec = out_dir + 'rec/'
# and what the files are supposed to be called
path_data = out_dir_data + 'data_scan_' + str(scannr).zfill(6) + '.ptyd'  # the file with the prepared data
path_dumps = out_dir_dumps + 'dump_scan_' + str(scannr).zfill(6) + '_%(engine)s_%(iterations)04d.ptyr'  # intermediate results
path_rec = out_dir_rec + 'rec_scan_' + str(scannr).zfill(6) + '_%(engine)s_%(iterations)04d.ptyr' 

By typing the following we can see that the parameter tree can be divided into 5 main parts:
'ptycho', 'io', 'scandata', 'scan', 'engine'. 

In [8]:
defaults_tree.children.keys()

odict_keys(['io', 'scandata', 'scan', 'engine', 'ptycho'])

The 'ptycho' part is the most top level part, and to define it in your script you type:

In [9]:
p = u.Param()  # Corresponding to "ptycho(Param)" in the documentation; https://ptycho.github.io/ptypy/rst/parameters.html#id4

To list the top level options for ptycho you can type the following:


In [10]:
defaults_tree['ptycho'].children.keys()

odict_keys(['verbose_level', 'data_type', 'run', 'frames_per_block', 'dry_run', 'ipython_kernel', 'io', 'scans', 'engines'])

And to get the documentation for each of these you can run:

In [11]:
for child in defaults_tree['ptycho'].children.values():
    print(child, '\n')

[verbose_level]
default = 'ERROR'
help = Verbosity level
type = str, int
doc = Verbosity level for information logging.
	- ``CRITICAL``: Only critical errors
	- ``ERROR``:    All errors
	- ``WARNING``:  Warning
	- ``INFO``:     Process Information
	- ``INSPECT``:  Object Information
	- ``DEBUG``:    Debug
userlevel = 0 

[data_type]
default = 'single'
help = Reconstruction floating number precision
type = str
doc = Reconstruction floating number precision (``'single'`` or
	``'double'``)
userlevel = 1 

[run]
default = None
help = Reconstruction identifier
type = str
doc = Reconstruction run identifier. If ``None``, the run name will
	be constructed at run time from other information.
userlevel = 0 

[frames_per_block]
default = 100000
help = Max number of frames per block of data
type = int
doc = This parameter determines the size of buffer arrays for GPUs.
	Reduce this number if you run out of memory on the GPU.
lowlim = 1
userlevel = 1 

[dry_run]
default = False
help = Dry run switc

This is the same information that is available in the full documentation page https://ptycho.github.io/ptypy/rst/parameters. As you can see the last 3 parameters is of type Param, meaning that these have further options in the form as a parameter tree. Let's define some of these in our script that we might want to change and initialize the deeper level of parameters!

In [ ]:
p.verbose_level = "info"  # It doesn't matter if you use upper or lower case here.
p.run = 'scan%d' % scannr
p.frames_per_block = 100000  # doesn't matter if its e.g. an odd number or exponential of 2 etc. but there will be more latency for smaller blocks since there will more transferring/copying between the cpus and gpus in that case.


p.io = u.Param()
p.scans = u.Param()
p.engines = u.Param()

If you want you could print **all** the options that are available under 'ptycho' by typing:
```
print(defaults_tree['ptycho'].to_string())
```
But this will produce a long output so we'll continue investigating the options level by level. To see the documentation for the options to p.io, p.scans and p.engines we do the same as above and then add some of these to our script:


In [13]:
for child in defaults_tree['ptycho.io'].children.values():
    print(child, '\n')

[home]
default = "./"
help = Base directory for all I/O
type = str
doc = home is the root directory for all input/output operations. All other path parameters that
	are relative paths will be relative to this directory. 

[rfile]
default = "recons/%(run)s/%(run)s_%(engine)s_%(iterations)04d.ptyr"
help = Reconstruction file name (or format string)
type = str
doc = Reconstruction file name or format string (constructed against runtime dictionary) 

[rformat]
default = "minimal"
help = Reconstruction file format
type = str
doc = Choose a reconstruction file format for after engine completion.
	- ``'minimal'``: Bare minimum of information
	- ``'dls'``:    Custom format for Diamond Light Source
choices = 'minimal','dls' 

[interaction]
default = None
help = ZeroMQ interactor options
type = Param
doc = Options for the communications server 

[autosave]
default = Param
help = Auto-save options
type = Param
doc = Options for automatic saving during reconstruction. 

[autoplot]
default = Param


In [ ]:
p.io.home = out_dir_rec
p.io.rfile = path_rec
p.io.interaction= u.Param()
p.io.autosave = u.Param()
p.io.autoplot = u.Param()

In [ ]:
for child in defaults_tree['ptycho.scans'].children.values():
    print(child, '\n')

for child in defaults_tree['ptycho.engines'].children.values():
    print(child, '\n')

As you can see we just got that the next level of parameters to put under 'scans' and 'engines' are wildcards. This is because you could define multiple scans and engines after each other. Usually you would just have one scan here but you might want to run different reconstruction engines after each other, e.g. first 'DM' and afterwards 'ML'. You can choose any name you want when defining the next level for these two but let's just go with 'scan00' and 'engine00' for now:

In [ ]:
p.scans.scan00 = u.Param()
p.engines.engine00 = u.Param()

To see the top level options for these we go back to the corresponding options that were listed directly under the defaults tree:

In [5]:
for child in defaults_tree['scan'].children.values():
    print(child, '\n')

[ScanModel]
default = 
help = 
type = Param 

[BlockScanModel]
default = 
help = 
type = Param 

[Vanilla]
default = 
help = 
type = Param 

[BlockVanilla]
default = 
help = 
type = Param 

[Full]
default = 
help = 
type = Param 

[BlockFull]
default = 
help = 
type = Param 

[OPRModel]
default = 
help = 
type = Param 

[BlockOPRModel]
default = 
help = 
type = Param 

[GradFull]
default = 
help = 
type = Param 

[BlockGradFull]
default = 
help = 
type = Param 

[Bragg3dModel]
default = 
help = 
type = Param 



While it's not visible here, all of these scanmodels are built on the 'ScanModel'. You will most likely just use 'BlockFull', or 'BlockGradFull' if using epie or other engines that reconstruct one position at a time serially. There is a current discussion on whether we will depricate the "Full" and "BlockFull" model. You can continue checking the options as before, 
```
for child in defaults_tree['scan.Full'].children.values():
    print(child, '\n')
```

or print all the options descending from a point by:
```
print(defaults_tree['scan.Full'].to_string())
```

In [6]:
for child in defaults_tree['scan.Full'].children.values():
    print(child, '\n')

[name]
default = Full
help = 
type = str
doc = 

[coherence]
default = 
help = Coherence parameters
type = Param
doc = 
userlevel = 
lowlim = 0 

[resolution]
default = None
help = Will force the reconstruction to adapt to the given resolution, this might lead to cropping/padding in diffraction space which could reduce performance.
type = None, float
doc = Half-period resolution given in [m]
userlevel = 0
lowlim = 0 

[tags]
default = ['dummy']
help = Comma seperated string tags describing the data input
type = list
doc = [deprecated?]
userlevel = 2 

[propagation]
default = farfield
help = Propagation type
type = str
doc = Either "farfield" or "nearfield"
userlevel = 1 

[ffttype]
default = scipy
help = FFT library
type = str
doc = Choose from "numpy", "scipy" or "fftw"
userlevel = 1 

[data]
default = 
help = Link to container for data preparation
type = @scandata.*
doc = 

[illumination]
default = 
help = Illumination parameters
type = Param, str 

[sample]
default = 
help = 
type =

\* Compatible options:\
&nbsp;&nbsp;&nbsp;&nbsp;model: engine

(Made a pull request for updating the verbose_level documentation)

In [19]:
print('defaults_tree: ', defaults_tree.children.keys())

print('ptycho: ', defaults_tree['ptycho'].children.keys())
print('ptycho.io: ', defaults_tree['ptycho.io'].children.keys())

print('scan: ', defaults_tree['scan'].children.keys())
print('scandata: ', defaults_tree['scandata'].children.keys())


defaults_tree:  odict_keys(['io', 'scandata', 'scan', 'engine', 'ptycho'])
ptycho:  odict_keys(['verbose_level', 'data_type', 'run', 'frames_per_block', 'dry_run', 'ipython_kernel', 'io', 'scans', 'engines'])
ptycho.io:  odict_keys(['home', 'rfile', 'rformat', 'interaction', 'autosave', 'autoplot', 'benchmark'])
scan:  odict_keys(['ScanModel', 'BlockScanModel', 'Vanilla', 'BlockVanilla', 'Full', 'BlockFull', 'OPRModel', 'BlockOPRModel', 'GradFull', 'BlockGradFull', 'Bragg3dModel'])
scandata:  odict_keys(['PtyScan', 'PtydScan', 'MoonFlowerScan', 'QuickScan', 'SimScan', 'NanomaxStepscanNov2018', 'NanomaxFlyscanMay2019', 'NanomaxStepscanSep2019', 'NanomaxFlyscanDec2019', 'NanomaxContrast'])


In [1]:
# Import all the neccessary modules
import ptypy
from ptypy import utils as u
from ptypy import defaults_tree

WARNING ptypy - Message Passaging for Python (mpi4py) not found.
    CPU-parallelization disabled.
    Install python-mpi4py via the package repositories or with `pip install --user mpi4py`


In [2]:
# Hardcoded info
scannr = 0    # Scan number created by the beamline. There is no need to specify the possible preceeding zeros of the scan number.

In [3]:
# General parameters
from ptypy import utils as u
p = u.Param()
p.verbose_level = 'interactive'
p.run = 'scan%d' % scannr
#p.min_frames_for_recon = start_frame  # default = 0, Minimum number of frames loaded before starting iterations
#p.frames_per_block = fpb              # default: 100000 #### should overrule min_frames at


In [4]:
print(defaults_tree['ptycho'].to_string())

[verbose_level]
default = 'ERROR'
help = Verbosity level
type = str, int
doc = Verbosity level for information logging.
	- ``CRITICAL``: Only critical errors
	- ``ERROR``:    All errors
	- ``WARNING``:  Warning
	- ``INFO``:     Process Information
	- ``INSPECT``:  Object Information
	- ``DEBUG``:    Debug
userlevel = 0

[data_type]
default = 'single'
help = Reconstruction floating number precision
type = str
doc = Reconstruction floating number precision (``'single'`` or
	``'double'``)
userlevel = 1

[run]
default = None
help = Reconstruction identifier
type = str
doc = Reconstruction run identifier. If ``None``, the run name will
	be constructed at run time from other information.
userlevel = 0

[frames_per_block]
default = 100000
help = Max number of frames per block of data
type = int
doc = This parameter determines the size of buffer arrays for GPUs.
	Reduce this number if you run out of memory on the GPU.
lowlim = 1
userlevel = 1

[dry_run]
default = False
help = Dry run switch
ty

In [5]:
import ptypy
from ptypy import defaults_tree
ptypy.load_ptyscan_module("nanomax")
print(defaults_tree['scandata.NanomaxContrast'].to_string())

bshuf filter already loaded, skip it.


[name]
default = NanomaxContrast
help = 
type = str

[dfile]
default = None
help = File path where prepared data will be saved in the ``ptyd`` format.
type = file
userlevel = 0

[chunk_format]
default = .chunk%02d
help = Appendix to saved files if save == 'link'
type = str
doc = 
userlevel = 2

[save]
default = None
help = Saving mode
type = str
doc = Mode to use to save data to file.
	<newline>
	- ``None``: No saving
	- ``'merge'``: attemts to merge data in single chunk **[not implemented]**
	- ``'append'``: appends each chunk in master \*.ptyd file
	- ``'link'``: appends external links in master \*.ptyd file and stores chunks separately
	<newline>
	in the path given by the link. Links file paths are relative to master file.
userlevel = 1

[auto_center]
default = None
help = Determine if center in data is calculated automatically
type = bool
doc = 
	- ``False``, no automatic centering
	- ``None``, only if :py:data:`center` is ``None``
	- ``True``, it will be enforced
userlevel = 0

[l

<h1>Parameters with notes</h1>

In [ ]:
p.io.home = out_dir_rec                           # where to save the final reconstructions
p.io.autosave.rfile = path_dumps                  # where to save the intermediate reconstructions and how to name them
p.io.rfile = path_rec                             # how to name those files for the final reconstructions

p.scans.scan00.name = 'BlockFull'                 # Corresponds to "scan.BlockFull.name(str)" in the documentation.
p.scans.scan00.data.name = 'LiveScan'             # Write the name of the PtyScan subclass. Corresponds to "scan.ScanModel.data.name(str)" in the documentation.
p.scans.scan_00.data.dfile = path_data            # once all data is collected, save it as .ptyd file
p.scans.scan00.data.shape = cropping              # size of the window of the diffraction patterns to be used in pixel
p.scans.scan00.data.center = (1284, 802)          # center of the diffraction pattern (y,x) in pixel or None -> auto
p.scans.scan00.data.distance = distance_m         # distance between sample and detector in [m]
p.scans.scan00.data.load_parallel = 'all'         # requires the center of the diffraction patterns to be given
p.scans.scan00.data.orientation = (False, False, False)    # Is dependent of the beamline detector, something that would be specific and known at the beamline.

p.scans.scan00.coherence.num_probe_modes = probe_modes                     # number of probe modes
p.scans.scan00.coherence.num_object_modes = 1                              # number of object modes

p.scans.scan00.illumination.model = None                                   # option 1: probe is initialized from a guess, could also be an np.array
p.scans.scan00.illumination.aperture.form = 'circ'                         # initial probe is a rectangle (KB focus)

p.scans.scan00.illumination.aperture.size = 28e-9                          # at the focus, actual beamsize. It's usually better to pick a value that is too large rather than too small.

p.scans.scan00.illumination.propagation.parallel = 1.*defocus_um*1e-6      # somehow this has to be negative to the basez axis
																	       # -> being downstream of the focus means negative distance
p.scans.scan00.illumination.propagation.parallel = 1. * defocus_um * 1e-6  # propagate the inital guess -> gives phase curvature

                                                                       
p.scans.scan00.illumination.diversity.noise = (.5, 1.0)               # default = (0.5, 1.0), Noise in each non-primary mode of the illumination.Can be either: - None : no noise - 2-tuple : noise in phase (amplitude (rms), minimum feature size) - 4-tuple : noise in phase & modulus (rms, mfs, rms_mod, mfs_mod)     # 'can be used for multiple modes' - maik | for some reason I get an error when I don't use these two..

p.scans.scan00.illumination.diversity.power = .1                      # default = 0.1 (>0.0, <1.0) , Power of modes relative to main mode (zero-layer) # 'can be used for multiple modes' - maik | for some reason I get an error when I don't use these two..


p.engines.engine00.numiter_contiguous = numitcont                     # Number of iterations without interruption
p.engines.engine00.obj_smooth_std = 10.                               # Default: None, Gaussian smoothing (pixel) of the current object prior to update.   If None, smoothing is deactivated. This smoothing can be used to reduce the amplitude of spurious pixels in the outer, least constrained areas of the object.
p.engines.engine00.overlap_converge_factor = 0.5                      # default = 0.05 (>0.0), Threshold for interruption of the inner overlap loop. The inner overlap loop refines the probe and the object simultaneously. This loop is escaped as soon as the overall change in probe, relative to the first iteration, is less than this value.
p.engines.engine00.overlap_max_iterations = 2                         # default = 10 (>1), Maximum of iterations for the overlap constraint inner loop
p.engines.engine00.probe_support = probe_support                      # non-zero probe area as fraction of the probe frame
p.engines.engine00.probe_update_start = 50                            # number of iterations before probe update starts

p.engines.engine00.position_refinement.start = 500                    # Number of iterations until position refinement starts
p.engines.engine00.position_refinement.stop = None                    # Number of iterations after which positon refinement stops, If None, stops after last iteration
p.engines.engine00.position_refinement.interval = 10                  # Frequency of position refinement
p.engines.engine00.position_refinement.nshifts = 4*64                 # Number of random shifts calculated in each position refinement step (has to be multiple of 4)
p.engines.engine00.position_refinement.amplitude = pxsize_m           # Distance from original position per random shift [m]
p.engines.engine00.position_refinement.max_shift = 100e-9             # Maximum distance from original position [m]
p.engines.engine00.position_refinement.record = True                  # record movement of positions


<h1>Current documentation and improvements</h1>

|Parameter|Documentation|Improvement|
|-----|-----|-----|
|<p>1scan.ScanModel.data.name(str)</p><br>   |<p>Name of the PtyScan subclass to use</p><p>default = None</p>||
|<p>1scan.ScanModel.data.name(str)</p><br>   |<p>Name of the PtyScan subclass to use</p><p>default = None</p>||
|<p>2scan.ScanModel.data.name(str)</p><br>   |<p>Name of the PtyScan subclass to use</p><p>default = None</p>||
|<p>2scan.ScanModel.data.name(str)</p><br>   |<p>Name of the PtyScan subclass to use</p><p>default = None</p>||



In [6]:
print(defaults_tree['scan'].to_string())

[ScanModel]
default = 
help = 
type = Param

[ScanModel.tags]
default = ['dummy']
help = Comma seperated string tags describing the data input
type = list
doc = [deprecated?]
userlevel = 2

[ScanModel.propagation]
default = farfield
help = Propagation type
type = str
doc = Either "farfield" or "nearfield"
userlevel = 1

[ScanModel.ffttype]
default = scipy
help = FFT library
type = str
doc = Choose from "numpy", "scipy" or "fftw"
userlevel = 1

[ScanModel.data]
default = 
help = Link to container for data preparation
type = @scandata.*
doc = 

[ScanModel.data.name]
default = 
help = Name of the PtyScan subclass to use
type = str

[ScanModel.illumination]
default = 
help = Container for probe initialization model
type = Param, str

[ScanModel.sample]
default = 
help = Container for sample initialization model
type = Param, str

[ScanModel.resample]
default = 1
help = Resampling fraction of the image frames w.r.t. diffraction frames
type = int, None
doc = A resampling of 2 means that the 

In [22]:
defaults_tree.__dict__

{'name': 'root',
 'parent': None,
 'children': OrderedDict([('io',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f56fb4f0>),
              ('scandata',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f5229f40>),
              ('scan',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f54d2e80>),
              ('engine',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f54d2190>),
              ('ptycho',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f55c2bb0>)]),
 'separator': '.',
 'required': ['default', 'help'],
 'optional': ['doc', 'type', 'userlevel', 'choices', 'uplim', 'lowlim'],
 'options_def': OrderedDict(),
 'num_id': 0,
 'options': {'default': '', 'help': '', 'type': 'Param'},
 '_all_options': {'default': '', 'help': '', 'type': 'Param'},
 'implicit': False}

In [21]:
#print(defaults_tree['engine.DM.position_refinement'].to_string())
print(defaults_tree['ptycho'].to_string())

defaults_tree.__dict__

[verbose_level]
default = 'ERROR'
help = Verbosity level
type = str, int
doc = Verbosity level for information logging.
	- ``CRITICAL``: Only critical errors
	- ``ERROR``:    All errors
	- ``WARNING``:  Warning
	- ``INFO``:     Process Information
	- ``INSPECT``:  Object Information
	- ``DEBUG``:    Debug
choices = ['CRITICAL', 'ERROR', 'WARNING', 'INFO', 'INSPECT', 'DEBUG']
userlevel = 0

[data_type]
default = 'single'
help = Reconstruction floating number precision
type = str
doc = Reconstruction floating number precision (``'single'`` or
	``'double'``)
choices = ['single', 'double']
userlevel = 1

[run]
default = None
help = Reconstruction identifier
type = str
doc = Reconstruction run identifier. If ``None``, the run name will
	be constructed at run time from other information.
userlevel = 0

[frames_per_block]
default = 100000
help = Max number of frames per block of data
type = int
doc = This parameter determines the size of buffer arrays for GPUs.
	Reduce this number if you run 

{'name': 'root',
 'parent': None,
 'children': OrderedDict([('io',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f56fb4f0>),
              ('scandata',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f5229f40>),
              ('scan',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f54d2e80>),
              ('engine',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f54d2190>),
              ('ptycho',
               <ptypy.utils.descriptor.EvalDescriptor at 0x14e1f55c2bb0>)]),
 'separator': '.',
 'required': ['default', 'help'],
 'optional': ['doc', 'type', 'userlevel', 'choices', 'uplim', 'lowlim'],
 'options_def': OrderedDict(),
 'num_id': 0,
 'options': {'default': '', 'help': '', 'type': 'Param'},
 '_all_options': {'default': '', 'help': '', 'type': 'Param'},
 'implicit': False}

In [8]:
import ptypy
from ptypy import defaults_tree
ptypy.load_ptyscan_module("nanomax")
print(defaults_tree['scandata.NanomaxContrast'].to_string())


[name]
default = NanomaxStepscanSep2019
help = 
type = str

[dfile]
default = None
help = File path where prepared data will be saved in the ``ptyd`` format.
type = file
userlevel = 0

[chunk_format]
default = .chunk%02d
help = Appendix to saved files if save == 'link'
type = str
doc = 
userlevel = 2

[save]
default = None
help = Saving mode
type = str
doc = Mode to use to save data to file.
	<newline>
	- ``None``: No saving
	- ``'merge'``: attemts to merge data in single chunk **[not implemented]**
	- ``'append'``: appends each chunk in master \*.ptyd file
	- ``'link'``: appends external links in master \*.ptyd file and stores chunks separately
	<newline>
	in the path given by the link. Links file paths are relative to master file.
userlevel = 1

[auto_center]
default = None
help = Determine if center in data is calculated automatically
type = bool
doc = 
	- ``False``, no automatic centering
	- ``None``, only if :py:data:`center` is ``None``
	- ``True``, it will be enforced
userlevel 